# Module 04: Evolutionary Computation & Edge of Chaos
Reconstruction of **John Holland's Schema Theorem**, **Mitchell, Hraber & Crutchfield (1993, 1994) Density Classification**, and **Reproducible Innovation Epoch Checkpoints**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# ------------------------------------------------------------------------------
# 1. RADIUS-3 CA ENGINE & PRE-COMPUTED BENCHMARK RULES (N=149, r=3)
# ------------------------------------------------------------------------------

def build_gkl_rule():
    '''
    Constructs the exact 128-bit Gacs-Kurdyumov-Levin (GKL) particle rule (r=3):
    If s_i == 0: majority(s_i, s_{i-1}, s_{i-3})
    If s_i == 1: majority(s_i, s_{i+1}, s_{i+3})
    '''
    rule = np.zeros(128, dtype=np.uint8)
    for idx in range(128):
        bits = [(idx >> (6 - k)) & 1 for k in range(7)]  # [i-3, i-2, i-1, i, i+1, i+2, i+3]
        s_i = bits[3]
        if s_i == 0:
            val = 1 if (bits[3] + bits[2] + bits[0]) >= 2 else 0
        else:
            val = 1 if (bits[3] + bits[4] + bits[6]) >= 2 else 0
        rule[idx] = val
    return rule

def build_local_majority_rule():
    '''Epoch 1 Strategy: Naive local majority over 7-cell neighborhood (blocks expand, interfaces stall).'''
    rule = np.zeros(128, dtype=np.uint8)
    for idx in range(128):
        pop = sum((idx >> k) & 1 for k in range(7))
        rule[idx] = 1 if pop >= 4 else 0
    return rule

def build_epoch2_block_expander_rule():
    '''Epoch 2 Strategy: Symmetry-broken block expansion (expands 1-blocks if >=3 ones, biased threshold).'''
    rule = np.zeros(128, dtype=np.uint8)
    for idx in range(128):
        bits = [(idx >> (6 - k)) & 1 for k in range(7)]
        pop = sum(bits)
        # Asymmetric rule breaking 0-1 symmetry (Mitchell et al., 1994 Epoch 2)
        if pop >= 5:
            rule[idx] = 1
        elif pop == 4 and bits[3] == 1:
            rule[idx] = 1
        elif pop == 3 and (bits[2] == 1 and bits[3] == 1 and bits[4] == 1):
            rule[idx] = 1
        else:
            rule[idx] = 0
    return rule

def simulate_r3_ca(rule_128, init_state, max_steps=150):
    '''Simulates a radius-3 CA on a 1D periodic lattice.'''
    width = len(init_state)
    grid = np.zeros((max_steps, width), dtype=np.uint8)
    grid[0] = init_state
    shifts = [-3, -2, -1, 0, 1, 2, 3]
    weights = np.array([64, 32, 16, 8, 4, 2, 1], dtype=np.int32)

    for t in range(max_steps - 1):
        idx = np.zeros(width, dtype=np.int32)
        for shift, w in zip(shifts, weights):
            idx += np.roll(grid[t], -shift) * w
        grid[t + 1] = rule_128[idx]
        if np.all(grid[t + 1] == grid[t]):
            grid[t + 1 :] = grid[t]
            break
    return grid

def evaluate_density_fitness(rule_128, n_ics=60, width=79, uniform_density=True, seed=42):
    '''Evaluates fraction of ICs correctly classified to all-0s or all-1s.'''
    rng = np.random.default_rng(seed)
    correct = 0
    for _ in range(n_ics):
        rho = rng.uniform(0.05, 0.95) if uniform_density else 0.5
        ic = (rng.random(width) < rho).astype(np.uint8)
        target = 1 if np.sum(ic) > (width / 2.0) else 0
        final_row = simulate_r3_ca(rule_128, ic, max_steps=2 * width)[-1]
        if np.all(final_row == target):
            correct += 1
    return correct / float(n_ics)

# ------------------------------------------------------------------------------
# 2. HOLLAND SCHEMA THEOREM & LIVE GA EVOLUTION SANDBOX
# ------------------------------------------------------------------------------

def holland_schema_lower_bound(m_t, f_schema, f_avg, defining_length, order_k, chrom_len=128, p_c=0.7, p_m=0.01):
    '''Computes Holland's Schema Theorem expected survival lower bound m(H, t+1).'''
    selection_gain = m_t * (f_schema / max(f_avg, 1e-9))
    crossover_survival = 1.0 - p_c * (defining_length / float(chrom_len - 1))
    mutation_survival = (1.0 - p_m) ** order_k
    return selection_gain * max(0.0, crossover_survival) * mutation_survival

def run_evolutionary_lab(checkpoint='Epoch 3/4: GKL Particle Logic', init_density=0.54, run_short_ga=False):
    rules_map = {
        'Epoch 1: Naive Local Majority': build_local_majority_rule(),
        'Epoch 2: Asymmetric Block Expander': build_epoch2_block_expander_rule(),
        'Epoch 3/4: GKL Particle Logic': build_gkl_rule(),
    }
    selected_rule = rules_map[checkpoint]

    # Generate reproducible test configuration with exact target density
    rng = np.random.default_rng(101)
    width = 149
    n_ones = int(round(init_density * width))
    ic = np.zeros(width, dtype=np.uint8)
    ic[:n_ones] = 1
    rng.shuffle(ic)

    spacetime = simulate_r3_ca(selected_rule, ic, max_steps=160)
    fit_uniform = evaluate_density_fitness(selected_rule, n_ics=50, width=79, uniform_density=True, seed=42)
    fit_unbiased = evaluate_density_fitness(selected_rule, n_ics=50, width=79, uniform_density=False, seed=42)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.0))
    fig.patch.set_facecolor('#F8FAFC')

    axes[0].imshow(spacetime, cmap='binary', interpolation='nearest', aspect='auto')
    true_cls = 'ALL-1s' if init_density > 0.5 else 'ALL-0s'
    axes[0].set_title(f'1. {checkpoint}\n(rho_0 = {init_density:.2f} -> Target: {true_cls})', fontsize=11, fontweight='bold')
    axes[0].set_xlabel('Lattice Site (N = 149)')
    axes[0].set_ylabel('Time Step t')

    # Panel 2: Fitness Comparison or Live Short GA Trajectory
    axes[1].set_facecolor('#F8FAFC')
    if not run_short_ga:
        names = ['Epoch 1\n(Majority)', 'Epoch 2\n(Block Exp.)', 'Epoch 3/4\n(GKL Particle)']
        u_scores = [evaluate_density_fitness(rules_map[k], 40, 69, True, 42) for k in rules_map]
        b_scores = [evaluate_density_fitness(rules_map[k], 40, 69, False, 42) for k in rules_map]
        x = np.arange(len(names))
        axes[1].bar(x - 0.17, u_scores, width=0.34, color='#2563EB', edgecolor='#0F172A', label='Uniform Density U[0,1]')
        axes[1].bar(x + 0.17, b_scores, width=0.34, color='#D97706', edgecolor='#0F172A', label='Unbiased (rho ~ 0.5)')
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(names, fontweight='bold')
        axes[1].set_ylim(0, 1.1)
        axes[1].set_ylabel('Classification Accuracy', fontweight='bold')
        axes[1].set_title('2. Innovation Epochs Benchmark (Mitchell et al., 1994)', fontsize=11, fontweight='bold')
        axes[1].legend(facecolor='white')
        axes[1].grid(axis='y', linestyle='--', alpha=0.4)
    else:
        # Fast live GA evolution (12 generations, N=49) to demonstrate selection dynamics
        ga_rng = np.random.default_rng(7)
        pop_size, gens = 20, 12
        pop = ga_rng.integers(0, 2, size=(pop_size, 128), dtype=np.uint8)
        best_hist, mean_hist = [], []
        for g in range(gens):
            fits = np.array([evaluate_density_fitness(ind, n_ics=20, width=49, uniform_density=True, seed=g) for ind in pop])
            best_hist.append(np.max(fits))
            mean_hist.append(np.mean(fits))
            elite_idx = np.argsort(fits)[-4:]
            next_pop = [pop[i].copy() for i in elite_idx]
            while len(next_pop) < pop_size:
                p1, p2 = pop[ga_rng.choice(elite_idx, 2, replace=True)]
                cut = ga_rng.integers(1, 127)
                child = np.concatenate([p1[:cut], p2[cut:]])
                mut_mask = ga_rng.random(128) < 0.02
                child[mut_mask] ^= 1
                next_pop.append(child)
            pop = np.array(next_pop)
        axes[1].plot(range(1, gens + 1), best_hist, marker='o', color='#0D9488', linewidth=2.2, label='Best Individual Fitness')
        axes[1].plot(range(1, gens + 1), mean_hist, marker='s', color='#64748B', linewidth=1.8, linestyle='--', label='Population Mean')
        axes[1].set_ylim(0, 1.05)
        axes[1].set_xlabel('Generation', fontweight='bold')
        axes[1].set_ylabel('Fitness (Accuracy)', fontweight='bold')
        axes[1].set_title('2. Live GA Evolution Trajectory (Seed=7, N=49)', fontsize=11, fontweight='bold')
        axes[1].legend(facecolor='white')
        axes[1].grid(True, linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.show()
    schema_next = holland_schema_lower_bound(m_t=4, f_schema=0.78, f_avg=0.52, defining_length=12, order_k=3)
    print('=' * 86)
    print(f'Selected Rule: {checkpoint} | Uniform-Density Acc: {fit_uniform:.2%} | Unbiased Acc: {fit_unbiased:.2%}')
    print(f'Holland Schema Theorem Example (m_t=4, f(H)/f_avg=1.50, delta=12, o(H)=3) => E[m(H, t+1)] >= {schema_next:.2f}')
    print('=' * 86)

widgets.interact(
    run_evolutionary_lab,
    checkpoint=widgets.Dropdown(options=['Epoch 1: Naive Local Majority', 'Epoch 2: Asymmetric Block Expander', 'Epoch 3/4: GKL Particle Logic'], value='Epoch 3/4: GKL Particle Logic', description='Checkpoint:'),
    init_density=widgets.FloatSlider(min=0.20, max=0.80, step=0.02, value=0.54, description='Init Density:'),
    run_short_ga=widgets.Checkbox(value=False, description='Run Live 12-Gen GA'),
);
